## Example 1 — the import, the version and a round trip of Python's richer types

In [ ]:
import datetime, decimal, fractions, importlib.metadata, ipaddress, uuid
import cbor2

print("cbor2", importlib.metadata.version("cbor2"), "-- core:", cbor2.dumps.__module__)
record = {
    "list": [1, 2.5, None, True, b"bytes"],
    "when": datetime.datetime(2026, 10, 2, 12, 30, tzinfo=datetime.timezone.utc),
    "decimal": decimal.Decimal("1.5"),
    "fraction": fractions.Fraction(1, 3),
    "uuid": uuid.UUID(int=5),
    "set": {1, 2, 3},
    "big": 2**80,
    "address": ipaddress.ip_address("192.0.2.1"),
}
encoded = cbor2.dumps(record)
decoded = cbor2.loads(encoded)
assert decoded == record, decoded
assert [type(decoded[k]) for k in record] == [type(v) for v in record.values()]
print(f"OK -- {len(record)} typed values round-trip through {len(encoded)} bytes of CBOR")

## Example 2 — exact wire bytes (RFC 8949 Appendix A) and canonical encoding

In [ ]:
import cbor2

VECTORS = [  # (value, the bytes RFC 8949 Appendix A gives for it)
    (0, "00"), (23, "17"), (24, "1818"), (1000000, "1a000f4240"), (-1, "20"), (-1000, "3903e7"),
    (18446744073709551616, "c249010000000000000000"),  # 2**64 as a tagged bignum
    ("a", "6161"), ("IETF", "6449455446"), (b"\x01\x02\x03\x04", "4401020304"),
    (False, "f4"), (True, "f5"), (None, "f6"),
]
for value, expected in VECTORS:
    assert cbor2.dumps(value).hex() == expected, (value, cbor2.dumps(value).hex(), expected)
    decoded = cbor2.loads(bytes.fromhex(expected))
    assert decoded == value and type(decoded) is type(value), (value, decoded)
assert cbor2.dumps([1, [2, 3], [4, 5]]).hex() == "8301820203820405"
# Canonical form: the shortest float that keeps the value, and map keys in a fixed order.
assert cbor2.dumps(1.5).hex() == "fb3ff8000000000000" and cbor2.dumps(1.5, canonical=True).hex() == "f93e00"
assert cbor2.dumps(100000.0, canonical=True).hex() == "fa47c35000"
assert cbor2.dumps({"b": 1, "a": 2}, canonical=True) == cbor2.dumps({"a": 2, "b": 1}, canonical=True)
print(f"OK -- {len(VECTORS)} RFC 8949 vectors encode and decode exactly; canonical floats and map order hold")

## Example 3 — your own types: an encoder hook and a tag hook

In [ ]:
import cbor2

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

def encode_point(encoder, value):  # called for a type the encoder does not know
    encoder.encode(cbor2.CBORTag(4000, [value.x, value.y]))

def decode_tag(tag, immutable):  # called for a tag the decoder has no built-in handling for
    return Point(*tag.value) if tag.tag == 4000 else tag

encoded = cbor2.dumps({"origin": Point(0, 0), "corner": Point(3, 4)}, default=encode_point)
decoded = cbor2.loads(encoded, tag_hook=decode_tag)
assert isinstance(decoded["corner"], Point) and (decoded["corner"].x, decoded["corner"].y) == (3, 4)
untouched = cbor2.loads(encoded)  # without the hook the tag itself comes back
assert untouched["corner"].tag == 4000 and list(untouched["corner"].value) == [3, 4], untouched["corner"]
hooked = cbor2.loads(cbor2.dumps({"k": 1}), object_hook=lambda mapping, immutable: sorted(mapping))
assert hooked == ["k"], hooked
print("OK -- a Point travels as tag 4000 and comes back through the tag hook:", vars(decoded["corner"]))

## Example 4 — streams, shared references and immutable decoding

In [ ]:
import io
import cbor2

stream = io.BytesIO()
for item in ([1, 2], "two", {"three": 3}):
    cbor2.dump(item, stream)
stream.seek(0)
assert [cbor2.load(stream) for _ in range(3)] == [[1, 2], "two", {"three": 3}]
cycle = [1]
cycle.append(cycle)  # a list that contains itself
try:
    cbor2.dumps(cycle)
    raise AssertionError("a cyclic structure was encoded without value sharing")
except cbor2.CBOREncodeValueError as error:
    refusal = str(error)
back = cbor2.loads(cbor2.dumps(cycle, value_sharing=True))
assert back[0] == 1 and back[1] is back, "the cycle did not survive the round trip"
frozen = cbor2.loads(cbor2.dumps({"k": [1, 2]}), immutable=True)
assert frozen == {"k": (1, 2)} and hash(frozen) == hash(frozen), type(frozen)
print(f"OK -- 3 items read back from one stream; a cycle is refused ('{refusal}') and kept with value_sharing")

## Example 5 — malformed input raises the documented errors; the command-line tool runs

In [ ]:
import subprocess, sys
import cbor2

BAD = {b"\x82\x01": cbor2.CBORDecodeEOF, b"\xa1": cbor2.CBORDecodeEOF, b"\x1f": cbor2.CBORDecodeError}
for data, expected in BAD.items():
    try:
        cbor2.loads(data)
        raise AssertionError(f"malformed input {data!r} was accepted")
    except cbor2.CBORDecodeError as error:
        assert type(error) is expected and isinstance(error, cbor2.CBORError), (data, type(error))
try:
    cbor2.dumps(object())
    raise AssertionError("an object with no encoder was accepted")
except cbor2.CBOREncodeError as error:
    assert "cannot encode" in str(error), error
# `python -m cbor2.tool` (installed as the `cbor2` command) turns CBOR into JSON.
tool = subprocess.run([sys.executable, "-m", "cbor2.tool"], input=cbor2.dumps({"a": [1, 2], "t": True}),
                      capture_output=True, timeout=120)
assert tool.returncode == 0, tool.stderr[-500:]
assert tool.stdout.strip() == b'{"a": [1, 2], "t": true}', tool.stdout
print("OK -- 3 malformed inputs and an unencodable object raise CBORError subclasses; the tool printed", tool.stdout.strip().decode())